# Funding

A set of tables for handling funding activities


In [ ]:
#| default_exp Funding

In [ ]:
#| export
#| hide
from biobib.table import Table, safe_float
from biobib.doc import Doc, NumbersDoc
from nbdev import show_doc
import pandas as pd
import time
import copy

In [ ]:
#| hide
from fastcore.test import *
from fastcore.test import *
from fastcore.basics import *
from fastcore.foundation import *

### Define a template for funding activities

Here is a template we can use for funding activities. It generates a longtable which can span multiple pages, and provides the years, funding source, proposal name and amount for each grant

In [ ]:
#| export
funding_biobib_template = r'''% UC Bio-bib Funding Table
% Created on \VAR{created}

\begin{longtable}{p{1.75cm}>{\raggedright}p{2.cm}p{5cm}|p{1.25cm}p{1.25cm}p{1.25cm}|p{0.75cm}p{0.75cm}}
Years & Agency/Source & Title & Total Amount & Total to UCSB & Personal Share & Role & New/Cont.\\
\hline 
\endfirsthead

\multicolumn{8}{c}%
{{\VAR{table_name} - continued from previous page }} \\ \\
Years & Agency/Source & Title & Total Amount & Total to UCSB & Personal Share & Role & New/Cont.\\
\hline 
\endhead

\\
\multicolumn{8}{c}%
{{ \VAR{table_name} continued on next page }} \\
\endfoot

\hline \hline
\endlastfoot

\BLOCK{for grant in items}
\VAR{grant.Year|tex_escape} & \VAR{grant.Source|tex_escape} & \VAR{grant.Title|tex_escape} & \VAR{grant['Total Amount']|format_currency|tex_escape} & \VAR{grant['Total to UCSB']|format_currency|tex_escape} & \BLOCK{if grant['Pooled Funds']} Pooled \BLOCK{else} \VAR{grant['Personal Share']|format_currency|tex_escape} \BLOCK{endif} & \VAR{grant.Role|tex_escape}  & \VAR{grant.Type|tex_escape} \\ 
\BLOCK{endfor}
& & \multicolumn{1}{r} {\bf Total New :} & \VAR{total_new['Total Amount']|tex_escape} & \VAR{total_new['Total to UCSB']|tex_escape} & \VAR{total_new['Total Personal Share']|tex_escape} & & \\
& & \multicolumn{1}{r} {\bf Total :} & \VAR{total['Total Amount']|tex_escape} & \VAR{total['Total to UCSB']|tex_escape} & \VAR{total['Total Personal Share']|tex_escape} & & \\

\end{longtable}

'''

In [ ]:
#| export
class Funding(Table):
    """ 
    A Funding class of Table for use in generating biobibs and CVs 
    """
    def __init__(
            self,
            doc:Doc=None, # Google sheet object
            worksheet:str='Grants',   # worksheet name in google sheet
            csv_file:str=None, # optional csv file (if using csv files) 
            table_name:str=None, # Will be determined from category.
            cumulative:bool=False, # Is this table cumulative? 
            template=funding_biobib_template # template content
    ): 
        filters = {
            'make_years': self.make_years
        }
        super().__init__(
            doc=doc,
            worksheet=worksheet, 
            table_name=table_name,
            csv_file=csv_file,
            filters=filters, 
            template=template)
        self.cumulative = cumulative
        self.df = self.clean_df()
    
    def total(self, new=False):
        if new:
            funds = self.df[self.df['Type'] == 'New']
        else:
            funds = self.df

        return {
            'Total Amount': '${:,.0f}'.format(
                funds['Total Amount'].replace('[\$,]', '', regex=True).apply(safe_float).sum()),
            'Total to UCSB': '${:,.0f}'.format(
                funds['Total to UCSB'].replace('[\$,]', '', regex=True).apply(safe_float).sum()),
            'Total Personal Share': '${:,.0f}'.format(
                funds['Personal Share'].replace('[\$,]', '', regex=True).apply(safe_float).sum())
        }


    def clean_df(self):
        df = copy.deepcopy(self.df)
        df = super().table_clean_df()
        # Fill NaN values in Funded column with 0 before converting to int
        df['Funded'] = df['Funded'].fillna(0)
        df = df[df['Funded'].astype(int) == 1]
        # Replace NaN with a 'nan' string for checking later
        df['Start Date'].fillna('nan', inplace=True)
        df['End Date'].fillna('nan', inplace=True)
        # Coerce the Start and End Dates to datetime
        df['Start Date'] = pd.to_datetime(df['Start Date'], errors='coerce')
        df['End Date'] = pd.to_datetime(df['End Date'], errors='coerce')
    
        df = df.sort_values(by=['Start Year', 'Total Amount'],
                            ascending=[True, False])
        return df

    def make_years(self, row):
        # If there is a start and end date, use those
        if row['Start Date'] != 'nan' and row['End Date'] != 'nan':
            return "{start}-{end}".format(
                start=row['Start Date'].year,
                end=row['End Date'].year)
        else:
        # Otherwise, use the years:
            return "{start}-{end}".format(
                start=row['Start Year'],
                end=row['End Year'])

    def render_template(self):
        rendered_tex = self.template.render(
            table_name=self.table_name,
            created=time.strftime("%Y-%m-%d"),
            items=list(self.df.to_dict('records')),
            total=self.total(new=False),
            total_new=self.total(new=True)
        )
        return rendered_tex


### Provide a data cleaning function for the funding table



In [ ]:
numbers_sheet_name = "CV.numbers"
doc_numbers = NumbersDoc(numbers_sheet_name)

In [ ]:
funding_table = Funding(
    doc=doc_numbers,worksheet='Grants',table_name='Funding',
    template=funding_biobib_template)

Cleaning table for  Funding


In [ ]:
test_eq(funding_table.table_name,'Funding')
test_eq(list(funding_table.df.columns),
        ['Start Year', 'End Year', 'Year', 'Start Date', 'End Date', 'Title', 'Source', 'Program', 
         'Award Number', 'Total Amount', 'Total to UCSB', 'Personal Share', 'Role', 'Summer Salary', 
         'Pooled Funds', 'Award Number', 'Duration', 'Yearly Amount', 'NSF', 'Federal, Non-NSF', 
         'Foundation', 'Other', 'Eval', 'Type', 'Value', 'Funded', 'Affiliation']
)

In [ ]:
print(funding_table.render_template())

% UC Bio-bib Funding Table
% Created on 2025-03-20

\begin{longtable}{p{1.75cm}>{\raggedright}p{2.cm}p{5cm}|p{1.25cm}p{1.25cm}p{1.25cm}|p{0.75cm}p{0.75cm}}
Years & Agency/Source & Title & Total Amount & Total to UCSB & Personal Share & Role & New/Cont.\\
\hline 
\endfirsthead

\multicolumn{8}{c}%
{{Funding - continued from previous page }} \\ \\
Years & Agency/Source & Title & Total Amount & Total to UCSB & Personal Share & Role & New/Cont.\\
\hline 
\endhead

\\
\multicolumn{8}{c}%
{{ Funding continued on next page }} \\
\endfoot

\hline \hline
\endlastfoot

2019-2022 & National Science Foundation & CNH2-L:  Linkages and Interactions Between Urban Food Security and Rural Agricultural Systems & \$1,600,000 & \$243,724 &  \$243,724  & Co-PI  & Cont \\ 
2019-2021 & Zegar Family Foundation & Developing and Implementing Novel Tools for Assessing Riparian Ecosystem Resilience at the Dangermond Preserve & \$321,583 & \$321,583 &  \$321,583  & PI  & Cont \\ 
2022-2023 & National Science Found

/var/folders/bs/x9tn9jz91cv6hb3q6p4djbmw0000gn/T/ipykernel_10616/2833404833.py:74: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  items=list(self.df.to_dict('records')),


In [ ]:
#| export
funding_cv_template = r'''% CV Funding Table
% Created on \VAR{created}

{\bf Total: }\VAR{total['Total Amount']|tex_escape}  \\

\begin{longtable}{p{1.75cm}>{\raggedright}p{2.cm}p{12cm}}
\endfirsthead

\multicolumn{3}{c}%
{{\VAR{table_name} - continued from previous page }} \\ \\
\endhead

\\
\multicolumn{3}{c}%
{{ \VAR{table_name} continued on next page }} \\
\endfoot


\endlastfoot


\BLOCK{for grant in items|reversed }
\VAR{grant.Year|tex_escape} & \VAR{grant.Source|tex_escape} & \VAR{grant.Title|tex_escape}, \VAR{grant.Role|tex_escape} (\VAR{grant['Total Amount']|format_currency|tex_escape}) \\ 
\BLOCK{endfor}

\end{longtable}
'''

In [ ]:
cv_table = Funding(
    doc=doc_numbers,worksheet='Grants',table_name='Funding',
    cumulative=True,
    template=funding_cv_template)

Cleaning table for  Funding


In [ ]:
print(cv_table.render_template())

% CV Funding Table
% Created on 2025-03-20

{\bf Total: }\$24,874,038  \\

\begin{longtable}{p{1.75cm}>{\raggedright}p{2.cm}p{12cm}}
\endfirsthead

\multicolumn{3}{c}%
{{Funding - continued from previous page }} \\ \\
\endhead

\\
\multicolumn{3}{c}%
{{ Funding continued on next page }} \\
\endfoot


\endlastfoot


2024-2025 & National Science Foundation & NSF I-Corps Hub: West Region NPI Supplement, PI (\$133,500) \\ 
2023-2025 & Department of Energy & Improving ESS approaches to evapotranspiration partitioning through data fusion, Co-PI (\$250,000) \\ 
2023-2025 & UC Office of the President & Broadening Participation and Seeding Climate Innovation for the Central Coast, PI (\$1,000,000) \\ 
2023-2028 & Department of Defense & The climate-food-urbanization nexus and the precursors of instability in Africa, Co-PI (\$3,073,150) \\ 
2022-2024 & Department of Defense & Generalizing plant functional responses to drought stress in the Southwest USA, Co-PI (\$178,284) \\ 
2022-2025 & Zegar F

/var/folders/bs/x9tn9jz91cv6hb3q6p4djbmw0000gn/T/ipykernel_10616/2833404833.py:74: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  items=list(self.df.to_dict('records')),


In [ ]:
cv_table.write_template(path='../tex/CV/')

/var/folders/bs/x9tn9jz91cv6hb3q6p4djbmw0000gn/T/ipykernel_10616/2833404833.py:74: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  items=list(self.df.to_dict('records')),


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()